# 02. Датасет для ручной разметки

## Введение

Этот notebook превращает candidate pool из `01_candidate_generation.ipynb` в понятный CSV для человека. Нам нужен не случайный набор строк, а смесь полезных случаев: пары с разных маркетплейсов, сложные negative-примеры, варианты фасовок и разные уровни похожести.

**Результат:** `labeling_<suffix>.csv` для одной категории или общий `labeling_sauces_coconut_oil_soap.csv` для обучения reranker.

## Оглавление

- [Введение](#введение)
- [Ход действий](#ход-действий)
- [0. Локальные настройки](#0-локальные-настройки)
- [1. Инструкция по ручной разметке](#1-инструкция-по-ручной-разметке)
- [2. Подготовка окружения](#2-подготовка-окружения)
- [3. Конфигурация выборки](#3-конфигурация-выборки)
- [4. Загрузка candidate CSV](#4-загрузка-candidate-csv)
- [5. Доступность страт](#5-доступность-страт)
- [6. Генерация labeling dataset](#6-генерация-labeling-dataset)
- [7. Сводка результата](#7-сводка-результата)
- [8. Проверка перед ручной работой](#8-проверка-перед-ручной-работой)
- [9. Итоговые выводы](#9-итоговые-выводы)

## Ход действий

1. Выбираем category-run или общий набор категорий для fine-tuning.
2. Загружаем candidate CSV из `01_candidate_generation.ipynb`.
3. Проверяем, какие страты реально доступны в данных.
4. Собираем сбалансированную выборку для человека: похожие пары, сложные negative-примеры и разные score-бакеты.
5. Сохраняем CSV и быстро проверяем, что разметчику будет понятно, какие две карточки сравнивать.


## 0. Локальные настройки

Здесь выбираем одну категорию для быстрого benchmark или несколько категорий для большого training set. Размер выборки, seed, batch id и score-бакеты задаются явно в notebook, чтобы разметку можно было воспроизвести.


In [ ]:
# === MY notebook settings ===
# Один run: ["soap"]. Fine-tuning set: ["sauces", "coconut_oil", "soap"].
MY_CATEGORY_RUNS = ['sauces', 'coconut_oil', 'soap']

# Только для single-run режима. None = стандартный candidates_<suffix>.csv.
MY_CANDIDATES_PATH = None

# None = стандартный labeling_<suffix>.csv или labeling_<all_suffixes>.csv.
MY_LABELING_PATH = None
MY_LABELING_TARGET_SIZE = 3000
MY_LABELING_RANDOM_STATE = 42
MY_LABELING_BATCH_ID = ''
MY_LABELING_PRESERVE_EXISTING_LABELS = True

# Ограничение разных брендов, чтобы датасет не забился слишком лёгкими negative-парами.
MY_LABELING_MAX_DIFFERENT_BRAND_SHARE = 0.2

# "quantile" = score-бакеты от текущего распределения. "absolute" = фиксированные пороги ниже.
MY_LABELING_SCORE_STRATIFICATION = 'quantile'
MY_LABELING_HIGH_TOP_SHARE = 0.25
MY_LABELING_EASY_BOTTOM_SHARE = 0.25
MY_LABELING_HIGH_THRESHOLD = 0.72
MY_LABELING_MEDIUM_LOWER = 0.5
MY_LABELING_MEDIUM_UPPER = 0.72
MY_LABELING_EASY_UPPER = 0.5


## 1. Инструкция по ручной разметке

Заполняйте колонку `label` одним из трёх значений: `exact_duplicate`, `different_product`, `uncertain`. Если решение спорное, в `notes` лучше написать короткую причину: это потом очень помогает разбирать ошибки модели.

- `exact_duplicate`: тот же базовый товар для ML-модели, даже если отличается фасовка или multipack. Например, `Соус соевой, 500 мл` и `Соус соевой, 500 мл - 2 шт`, если бренд/тип/вкус совпадают.
- `different_product`: карточки похожи, но это разные товары. Например, один бренд и вес, но разные вкусы: барбекю против сладкого чили.
- `uncertain`: без карточки, состава или ручной проверки нельзя честно решить.

Колонки `labeling_stratum`, `is_cross_marketplace_pair`, `is_hard_negative_candidate` и `is_pack_variant_candidate` — это подсказки о том, почему пара попала в выборку, а не правильный ответ. Pack-variant пары размечаем как `exact_duplicate`, если это тот же базовый товар; конкретная фасовка будет отделена позже deterministic pack-правилами.

Для большого датасета под reranker оставьте `MY_CATEGORY_RUNS = ["sauces", "coconut_oil", "soap"]` и `MY_LABELING_TARGET_SIZE = 3000`: notebook поделит размер между категориями и сохранит разные типы пар внутри каждой.


## 2. Подготовка окружения

Подключаем `research.dedup` helpers и настраиваем pandas. В этой секции нет чтения CSV и нет записи файлов.


In [ ]:
from dataclasses import replace
from pathlib import Path
import hashlib
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from research.dedup import (
    LabelingSamplingConfig,
    add_brand_relation_flags,
    add_cross_marketplace_flags,
    add_pack_variant_flags,
    labeling_pair_keys,
    labeling_score_strata_masks,
    merge_preserved_labeling_rows,
    select_preserved_labeling_rows,
    split_labeling_target_size,
    stratified_labeling_sample,
    resolve_category_run,
    resolve_run_paths,
)

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 140)


## 3. Конфигурация выборки

Нормализуем список категорий, делим общий target size и фиксируем seed. Для multi-category датасета каждая категория получает свой стабильный random state, поэтому запуск можно повторить и получить тот же CSV.


In [ ]:
def parse_category_runs(raw_runs) -> list:
    '''Преобразует строку или список пользовательских category-runs в уникальные run-конфиги.'''
    if isinstance(raw_runs, str):
        requested_runs = [part.strip() for part in raw_runs.replace(";", ",").split(",") if part.strip()]
    else:
        requested_runs = [str(part).strip() for part in raw_runs if str(part).strip()]
    if not requested_runs:
        requested_runs = [None]

    runs = []
    seen_slugs = set()
    for requested_run in requested_runs:
        run = resolve_category_run(requested_run)
        if run.slug in seen_slugs:
            continue
        runs.append(run)
        seen_slugs.add(run.slug)
    return runs


def stable_batch_offset(batch_id: str) -> int:
    '''Даёт стабильное смещение seed для независимых batches ручной разметки.'''
    if not batch_id:
        return 0
    digest = hashlib.sha1(batch_id.encode("utf-8")).hexdigest()[:8]
    return int(digest, 16) % 1_000_000


CATEGORY_RUNS = parse_category_runs(MY_CATEGORY_RUNS)
RUN_PATHS_BY_SLUG = {run.slug: resolve_run_paths(PROJECT_ROOT, run) for run in CATEGORY_RUNS}
IS_MULTI_RUN = len(CATEGORY_RUNS) > 1


def candidates_path_for_run(run) -> Path:
    '''Возвращает candidate CSV для одного run с учётом single-run override.'''
    if MY_CANDIDATES_PATH and not IS_MULTI_RUN:
        return Path(MY_CANDIDATES_PATH).expanduser()
    return RUN_PATHS_BY_SLUG[run.slug].candidates_path


def default_labeling_path() -> Path:
    '''Строит стандартный путь labeling CSV для single-run или multi-run режима.'''
    if not IS_MULTI_RUN:
        return RUN_PATHS_BY_SLUG[CATEGORY_RUNS[0].slug].labeling_path
    suffix = "_".join(run.artifact_suffix for run in CATEGORY_RUNS)
    return PROJECT_ROOT / "research" / "dedup" / "data" / f"labeling_{suffix}.csv"


LABELING_PATH = Path(MY_LABELING_PATH).expanduser() if MY_LABELING_PATH else default_labeling_path()
TARGET_SAMPLE_SIZE = int(MY_LABELING_TARGET_SIZE)
RANDOM_STATE = int(MY_LABELING_RANDOM_STATE)
BATCH_ID = str(MY_LABELING_BATCH_ID or "").strip()
BATCH_SEED_OFFSET = stable_batch_offset(BATCH_ID)
EFFECTIVE_RANDOM_STATE = (RANDOM_STATE + BATCH_SEED_OFFSET) % (2**32 - 1)
PRESERVE_EXISTING_LABELS = bool(MY_LABELING_PRESERVE_EXISTING_LABELS)
MAX_DIFFERENT_BRAND_SHARE = float(MY_LABELING_MAX_DIFFERENT_BRAND_SHARE)
SCORE_STRATIFICATION = str(MY_LABELING_SCORE_STRATIFICATION or "quantile").strip() or "quantile"
HIGH_SIMILARITY_TOP_SHARE = float(MY_LABELING_HIGH_TOP_SHARE)
EASY_NEGATIVE_BOTTOM_SHARE = float(MY_LABELING_EASY_BOTTOM_SHARE)
HIGH_SIMILARITY_THRESHOLD = float(MY_LABELING_HIGH_THRESHOLD)
MEDIUM_SIMILARITY_LOWER = float(MY_LABELING_MEDIUM_LOWER)
MEDIUM_SIMILARITY_UPPER = float(MY_LABELING_MEDIUM_UPPER)
EASY_NEGATIVE_UPPER = float(MY_LABELING_EASY_UPPER)


def make_sampling_config(target_size: int, *, run_offset: int = 0) -> LabelingSamplingConfig:
    '''Собирает конфиг стратифицированной выборки для одного category-run.'''
    return LabelingSamplingConfig(
        target_size=target_size,
        random_state=EFFECTIVE_RANDOM_STATE + run_offset * 1000,
        score_stratification=SCORE_STRATIFICATION,
        high_similarity_top_share=HIGH_SIMILARITY_TOP_SHARE,
        easy_negative_bottom_share=EASY_NEGATIVE_BOTTOM_SHARE,
        high_similarity_threshold=HIGH_SIMILARITY_THRESHOLD,
        medium_similarity_lower=MEDIUM_SIMILARITY_LOWER,
        medium_similarity_upper=MEDIUM_SIMILARITY_UPPER,
        easy_negative_upper=EASY_NEGATIVE_UPPER,
        max_different_brand_share=MAX_DIFFERENT_BRAND_SHARE,
    )


RUN_TARGET_SIZES = split_labeling_target_size(TARGET_SAMPLE_SIZE, len(CATEGORY_RUNS))
RUN_SAMPLING_CONFIGS = {
    run.slug: make_sampling_config(target_size, run_offset=idx)
    for idx, (run, target_size) in enumerate(zip(CATEGORY_RUNS, RUN_TARGET_SIZES, strict=True))
}
sampling_config = make_sampling_config(TARGET_SAMPLE_SIZE)

print(
    "Category runs: "
    + ", ".join(f"{run.slug} — {run.display_name}; project: {run.project_name}" for run in CATEGORY_RUNS)
)
print(f"Labeling path: {LABELING_PATH}")
print(f"Target sample size: {sampling_config.target_size}")
print(f"Random state: {RANDOM_STATE}")
print(f"Batch id: {BATCH_ID or '<default>'}; effective random state: {sampling_config.random_state}")
print(f"Preserve existing labels: {PRESERVE_EXISTING_LABELS}")
print(f"Max different-brand share: {sampling_config.max_different_brand_share:.0%}")
print(f"Score stratification: {sampling_config.score_stratification}")
if sampling_config.score_stratification == "quantile":
    print(
        "Score buckets: "
        f"top {sampling_config.high_similarity_top_share:.0%} -> high_similarity; "
        f"bottom {sampling_config.easy_negative_bottom_share:.0%} -> random_easy_negative"
    )
else:
    print(
        "Absolute score thresholds: "
        f"high >= {sampling_config.high_similarity_threshold}; "
        f"medium [{sampling_config.medium_similarity_lower}, {sampling_config.medium_similarity_upper}); "
        f"easy < {sampling_config.easy_negative_upper}"
    )


display(
    pd.DataFrame(
        [
            {
                "category_run": run.slug,
                "category_name": run.display_name,
                "project_name": run.project_name,
                "candidate_path": str(candidates_path_for_run(run)),
                "sample_target": RUN_SAMPLING_CONFIGS[run.slug].target_size,
                "random_state": RUN_SAMPLING_CONFIGS[run.slug].random_state,
                "max_different_brand_share": RUN_SAMPLING_CONFIGS[run.slug].max_different_brand_share,
            }
            for run in CATEGORY_RUNS
        ]
    )
)


## 4. Загрузка candidate CSV

Для каждого category-run читаем `candidates_<suffix>.csv`, добавляем service-флаги и сохраняем короткую сводку по числу доступных пар.


In [ ]:
candidate_frames = {}
load_rows = []

for run in CATEGORY_RUNS:
    candidates_path = candidates_path_for_run(run)
    if not candidates_path.exists():
        raise FileNotFoundError(
            f"Не найден {candidates_path}. Сначала выполните notebooks/01_candidate_generation.ipynb "
            f"для category-run {run.slug}."
        )

    frame = pd.read_csv(candidates_path)
    frame = add_cross_marketplace_flags(add_pack_variant_flags(frame))
    frame["category_run"] = run.slug
    frame["category_name"] = run.display_name
    frame["project_name"] = run.project_name
    candidate_frames[run.slug] = frame
    load_rows.append(
        {
            "category_run": run.slug,
            "candidate_pairs": len(frame),
            "sample_target": RUN_SAMPLING_CONFIGS[run.slug].target_size,
            "candidate_path": str(candidates_path),
        }
    )

load_summary = pd.DataFrame(load_rows)
print(f"Loaded candidates: {load_summary['candidate_pairs'].sum():,} pairs across {len(candidate_frames)} category-run(s)")
display(load_summary)
display(candidate_frames[CATEGORY_RUNS[0].slug].head(5))


## 5. Доступность страт

Перед sampling проверяем, сколько пар реально доступно в каждой страте. Это честный момент контроля: если где-то мало hard negatives или pack variants, это видно до начала ручной разметки.


In [ ]:
availability_rows = []
score_bucket_ranges = []

for run in CATEGORY_RUNS:
    candidates = candidate_frames[run.slug]
    run_config = RUN_SAMPLING_CONFIGS[run.slug]
    score = pd.to_numeric(candidates["baseline_similarity_score"], errors="coerce").fillna(0.0)
    score_masks = labeling_score_strata_masks(candidates, run_config)
    availability_rows.extend(
        [
            {
                "category_run": run.slug,
                "stratum": "cross_marketplace_candidate",
                "available_pairs": int(candidates["is_cross_marketplace_pair"].fillna(False).sum()),
            },
            {
                "category_run": run.slug,
                "stratum": "hard_negative_candidate",
                "available_pairs": int(candidates["is_hard_negative_candidate"].fillna(False).sum()),
            },
            {
                "category_run": run.slug,
                "stratum": "pack_variant_candidate",
                "available_pairs": int(candidates["is_pack_variant_candidate"].fillna(False).sum()),
            },
            {
                "category_run": run.slug,
                "stratum": "high_similarity",
                "available_pairs": int(score_masks["high_similarity"].sum()),
            },
            {
                "category_run": run.slug,
                "stratum": "medium_similarity",
                "available_pairs": int(score_masks["medium_similarity"].sum()),
            },
            {
                "category_run": run.slug,
                "stratum": "random_easy_negative",
                "available_pairs": int(score_masks["random_easy_negative"].sum()),
            },
        ]
    )

    for stratum in ["high_similarity", "medium_similarity", "random_easy_negative"]:
        values = score[score_masks[stratum]]
        score_bucket_ranges.append(
            {
                "category_run": run.slug,
                "stratum": stratum,
                "pairs": int(len(values)),
                "min_score": values.min() if len(values) else pd.NA,
                "median_score": values.median() if len(values) else pd.NA,
                "max_score": values.max() if len(values) else pd.NA,
            }
        )

availability = pd.DataFrame(availability_rows)
display(availability)
display(pd.DataFrame(score_bucket_ranges))


## 6. Генерация labeling dataset

Если `LABELING_PATH` уже существует, сохраняем заполненные labels и добираем только недостающий остаток. Так ручная работа не теряется при повторной генерации.


In [ ]:
existing_labeling_df = pd.DataFrame()
preserved_labeling_df = pd.DataFrame()
if PRESERVE_EXISTING_LABELS and LABELING_PATH.exists():
    existing_labeling_df = pd.read_csv(LABELING_PATH)
    preserved_labeling_df = select_preserved_labeling_rows(existing_labeling_df)
    if not preserved_labeling_df.empty:
        backup_path = LABELING_PATH.with_name(f"{LABELING_PATH.stem}_preserved_labels.csv")
        preserved_labeling_df.to_csv(backup_path, index=False)
        print(f"Preserved labeled rows: {len(preserved_labeling_df):,}")
        print(f"Saved preserved-label backup: {backup_path}")
    else:
        print("Existing labeling CSV found, but no filled labels to preserve.")
else:
    print("No existing labels loaded for preservation.")


def preserved_rows_for_run(run) -> pd.DataFrame:
    '''Выбирает уже размеченные строки, относящиеся к конкретному category-run.'''
    if preserved_labeling_df.empty:
        return preserved_labeling_df.copy()
    if "category_run" in preserved_labeling_df.columns:
        return preserved_labeling_df[preserved_labeling_df["category_run"].fillna("").astype(str).eq(run.slug)].copy()
    if not IS_MULTI_RUN:
        return preserved_labeling_df.copy()
    return preserved_labeling_df.head(0).copy()


def fill_run_metadata(frame: pd.DataFrame, run) -> pd.DataFrame:
    '''Гарантирует category/project metadata в новых и сохранённых строках.'''
    result = frame.copy()
    for column, value in {
        "category_run": run.slug,
        "category_name": run.display_name,
        "project_name": run.project_name,
    }.items():
        if column not in result.columns:
            result[column] = value
        else:
            result[column] = result[column].where(result[column].notna(), "").astype(str).replace("", value)
    return result


labeling_frames = []
for idx, run in enumerate(CATEGORY_RUNS):
    run_config = RUN_SAMPLING_CONFIGS[run.slug]
    run_preserved = fill_run_metadata(preserved_rows_for_run(run), run)
    run_preserved = add_brand_relation_flags(run_preserved) if not run_preserved.empty else run_preserved
    preserved_pair_keys = labeling_pair_keys(run_preserved)
    preserved_diff_count = int(run_preserved.get("is_different_brand_pair", pd.Series(dtype=bool)).fillna(False).sum())
    max_final_diff_count = int(run_config.target_size * run_config.max_different_brand_share)
    fresh_target_size = max(0, run_config.target_size - len(run_preserved))
    fresh_diff_budget = max(0, max_final_diff_count - preserved_diff_count)
    fresh_config = replace(
        run_config,
        target_size=fresh_target_size,
        max_different_brand_count=fresh_diff_budget,
    )

    run_fresh_labeling = stratified_labeling_sample(
        candidate_frames[run.slug],
        fresh_config,
        excluded_pair_keys=preserved_pair_keys,
    )
    run_fresh_labeling = fill_run_metadata(run_fresh_labeling, run)
    run_labeling = merge_preserved_labeling_rows(
        run_preserved,
        run_fresh_labeling,
        target_size=run_config.target_size,
        random_state=run_config.random_state + 20_000,
    )
    labeling_frames.append(run_labeling)

labeling_df = pd.concat(labeling_frames, ignore_index=True) if labeling_frames else pd.DataFrame()
if not labeling_df.empty:
    labeling_df = labeling_df.sample(frac=1, random_state=sampling_config.random_state + 50_000).reset_index(drop=True)

export_columns = [
    "label",
    "notes",
    "category_run",
    "category_name",
    "project_name",
    "labeling_stratum",
    "brand_relation",
    "is_different_brand_pair",
    "raw_record_id_a",
    "raw_record_id_b",
    "marketplace_a",
    "marketplace_b",
    "marketplaces_a",
    "marketplaces_b",
    "sku_a",
    "sku_b",
    "title_a",
    "title_b",
    "brand_a",
    "brand_b",
    "subcategory_a",
    "subcategory_b",
    "subcategory_relation",
    "unit_amount_a",
    "unit_amount_b",
    "total_amount_a",
    "total_amount_b",
    "multipack_count_a",
    "multipack_count_b",
    "embedding_similarity_score",
    "candidate_rank",
    "candidate_source",
    "blocking_scope",
    "baseline_similarity_score",
    "is_cross_marketplace_pair",
    "is_hard_negative_candidate",
    "is_pack_variant_candidate",
]
labeling_df = labeling_df[[column for column in export_columns if column in labeling_df.columns]].copy()

LABELING_PATH.parent.mkdir(parents=True, exist_ok=True)
labeling_df.to_csv(LABELING_PATH, index=False)
print(f"Saved labeling dataset: {LABELING_PATH}")
print(f"Rows saved: {len(labeling_df):,}")
print(f"Rows with preserved labels: {labeling_df['label'].fillna('').astype(str).str.strip().ne('').sum():,}")


## 7. Сводка результата

Смотрим баланс category-runs, страт, cross-marketplace пар и пар с разными брендами. Это последний быстрый контроль перед передачей CSV на разметку.


In [ ]:
category_stats = (
    labeling_df.groupby(["category_run", "category_name", "project_name"])
    .agg(
        pairs=("category_run", "size"),
        labeled_pairs=("label", lambda s: s.fillna("").astype(str).str.strip().ne("").sum()),
        cross_marketplace_pairs=("is_cross_marketplace_pair", "sum"),
        hard_negative_pairs=("is_hard_negative_candidate", "sum"),
        pack_variant_pairs=("is_pack_variant_candidate", "sum"),
        different_brand_pairs=("is_different_brand_pair", "sum"),
    )
    .reset_index()
    .sort_values("category_run")
)
category_stats["different_brand_share"] = category_stats["different_brand_pairs"] / category_stats["pairs"]
display(category_stats)

strata_stats = (
    labeling_df.groupby(["category_run", "labeling_stratum"])
    .agg(
        pairs=("labeling_stratum", "size"),
        cross_marketplace_pairs=("is_cross_marketplace_pair", "sum"),
        different_brand_pairs=("is_different_brand_pair", "sum"),
    )
    .reset_index()
    .sort_values(["category_run", "labeling_stratum"])
)
display(strata_stats)

brand_relation_stats = (
    labeling_df.groupby(["category_run", "brand_relation"])
    .size()
    .reset_index(name="pairs")
    .sort_values(["category_run", "brand_relation"])
)
display(brand_relation_stats)

score_by_stratum = labeling_df.groupby(["category_run", "labeling_stratum"])["baseline_similarity_score"].agg(
    pairs="count",
    min="min",
    median="median",
    max="max",
).reset_index()
display(score_by_stratum)

cross_marketplace_summary = (
    labeling_df.assign(
        same_marketplace_pair=~labeling_df["is_cross_marketplace_pair"].fillna(False)
    )
    .groupby("category_run")
    .agg(
        rows_total=("category_run", "size"),
        cross_marketplace_pairs=("is_cross_marketplace_pair", "sum"),
        same_marketplace_pairs=("same_marketplace_pair", "sum"),
    )
    .reset_index()
)
display(cross_marketplace_summary)


## 8. Проверка перед ручной работой

Проверяем, что экспорт содержит только ожидаемые labels и что каждая страта выглядит как нужный тип ручного контроля. После этой ячейки CSV уже можно отдавать разметчикам.


In [ ]:
labels = labeling_df["label"].fillna("").astype(str).str.strip()
valid_labels = {"", "exact_duplicate", "different_product", "uncertain", "same_product_different_pack"}
assert labels.isin(valid_labels).all(), "В label есть неизвестные значения"
assert labeling_df["notes"].fillna("").astype(str).notna().all()
assert len(labeling_df) <= sampling_config.target_size or labels.ne("").sum() > sampling_config.target_size

for run in CATEGORY_RUNS:
    run_rows = labeling_df[labeling_df["category_run"].eq(run.slug)]
    if run_rows.empty:
        continue
    diff_count = int(run_rows["is_different_brand_pair"].fillna(False).sum())
    diff_limit = int(RUN_SAMPLING_CONFIGS[run.slug].target_size * MAX_DIFFERENT_BRAND_SHARE)
    preserved_run_diff = int(
        preserved_rows_for_run(run).pipe(add_brand_relation_flags)["is_different_brand_pair"].fillna(False).sum()
    ) if not preserved_rows_for_run(run).empty else 0
    assert diff_count <= diff_limit or preserved_run_diff > diff_limit

for row in strata_stats[["category_run", "labeling_stratum"]].itertuples(index=False):
    print(f"{row.category_run} / {row.labeling_stratum}")
    display(
        labeling_df[
            labeling_df["category_run"].eq(row.category_run)
            & labeling_df["labeling_stratum"].eq(row.labeling_stratum)
        ]
        .sort_values("baseline_similarity_score", ascending=False)
        .head(3)
    )


## 9. Итоговые выводы

Главный артефакт notebook — CSV в `LABELING_PATH`. Для одиночного run это `labeling_<suffix>.csv`, для multi-run — общий файл вроде `labeling_sauces_coconut_oil_soap.csv`. После ручной разметки он становится gold-set для сравнения matching-моделей и training set для fine-tuning reranker.

Если `LABELING_PATH` уже существует, notebook бережно сохраняет заполненные `label`-строки, пишет backup `*_preserved_labels.csv` и добирает только незаполненный остаток. Для разных разметчиков используйте разные `MY_LABELING_BATCH_ID` или `MY_LABELING_RANDOM_STATE`.
